# Historical news audit
Read-only reproduction. Run from the repository root or set root explicitly. No network calls or model evaluation.

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd

root = Path.cwd()
while not (root / "research/price_forecasting/news_archive.py").exists():
    if root == root.parent:
        raise RuntimeError("Repository root not found")
    root = root.parent
files = sorted((root / "data/news").rglob("*.jsonl"))
d = pd.DataFrame(
    [json.loads(line) for f in files for line in f.read_text().splitlines() if line.strip()]
)
p = pd.to_datetime(d.published_at, utc=True, errors="raise")
c = pd.to_datetime(d.collected_at, utc=True, errors="raise")
print("Records:", len(d))
print(d.groupby(["ticker", "provider"]).size())
print("Annual counts:", p.dt.year.value_counts().sort_index().to_dict())
print("Exact key duplicates:", d.duplicated(["provider", "id", "ticker"]).sum())
print("Exact headline duplicates:", d.duplicated(["ticker", "headline", "published_at"]).sum())
print("Missing:", d.isna().sum().to_dict())
print("Publication after collection:", (p > c).sum())
print("Lag days:", ((c - p).dt.total_seconds() / 86400).describe())
window = (p >= pd.Timestamp("2023-03-30", tz="UTC")) & (p < pd.Timestamp("2024-12-10", tz="UTC"))
print("Validation records:", d.loc[window].groupby("provider").size())
for f in files:
    manifest = json.loads(f.with_suffix(".manifest.json").read_text())
    print(
        f.name,
        "byte_match",
        hashlib.sha256(f.read_bytes()).hexdigest() == manifest["sha256"],
        "LF_match",
        hashlib.sha256(f.read_text().encode()).hexdigest() == manifest["sha256"],
    )